# 08 - Primeiro trabalho

Em dupla ou trio, faça de acordo com a especificação do trabalho, usando os conceitos de método da bisseção, métodos de ponto fixo, métodos de Gauss e resolução de sistemas não lineares.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana8
```

Faça o checkout nessa nova branch:

```bash
git checkout semana8
```

<hr />

## Atividade 1
Para economizar energia elétrica, um agricultor implantou um sistema de painel solar na fazenda para alimentar uma bomba d’água, que faz a irrigação das plantações.  
A placa negra utilizada fica exposta ao sol e ao vento. Dessa forma, para gerar eletricidade, a irradiação solar $E$ (em $\text{W/m}^2$) sobre a placa tem que ser maior que a perda de calor por radiação e por condução, dada pela temperatura atmosférica $K$ (em Kelvin).

A equação é dada por:

$$
E = 5,67 \times 10^{-8} \, T^4 + 0,4(T - K)
$$

Determine a temperatura mínima da placa, dada a medição da irradiação e da temperatura do ar, sabendo que o valor médio diário é $E = 500,125$ e $K = 272,975$.

**Sugestão:** usar métodos da bissecção ou ponto fixo.  

Definindo $f(T)=5,67\times10^{-8}T^4+0,4(T-272,975)-500,125$, a raiz calculada pelo método da bissecção é:

$$
T \approx 304{,}509195432981\,K
$$

Para $T>0$, $f'(T)=4(5,67\times10^{-8})T^3+0,4>0$; portanto, essa raiz positiva é única e corresponde à temperatura mínima que atende ao balanço indicado.

Substituindo esse valor na equação, obtém-se $f(T)\approx 0$ (resíduo numérico da ordem de $10^{-12}\,\text{W/m}^2$). O resultado 304,56801011987010952 K que aparece no enunciado não é compatível com os dados: nele, o lado direito da equação vale aproximadamente $500,525278\,\text{W/m}^2$, uma diferença de $0,400278\,\text{W/m}^2$ em relação a $E=500,125\,\text{W/m}^2$. Portanto, foi mantida a equação e os dados informados e corrigida a resposta, sem alterar os valores do problema.


In [ ]:
import os
import sys

import numpy as np

sys.path.append(os.path.abspath(os.path.join('..')))
from utils.algoritmos import bissecao, seidel


In [ ]:
E = 500.125
K = 272.975
f = lambda T: 5.67e-8 * T**4 + 0.4 * (T - K) - E

T, iteracoes = bissecao(f, 250, 350, TOL=1e-12)
print(f"Temperatura mínima: {T:.12f} K")
print(f"Resíduo: {f(T):.3e} W/m²")
print(f"Iterações: {iteracoes}")


## Atividade 2
Em um grande hospital, para evitar intermitência no sistema elétrico dos diversos equipamentos médicos, instalou-se três reatores de alta performance em série, com tensão nominal $R$ em função da potência máxima $P$ de operação dos enrolamentos de controle:

$$
17R_1 - 2R_2 - 3R_3 = P_1 = 500
$$

$$
-5R_1 + 21R_2 - 2R_3 = P_2 = 200
$$

$$
-5R_1 - 5R_2 + 22R_3 = P_3 = 300
$$

Calcular as tensões nominais de cada reator.

**Sugestão:** usar o método de Gauss-Seidel.

O sistema foi escrito na forma $A\,R=P$ e resolvido iterativamente até que a diferença entre aproximações sucessivas fosse menor que $10^{-12}$. As tensões encontradas são:

$$
R_1 \approx 36{,}560816557981\,V,\qquad
R_2 \approx 20{,}768358378225\,V,\qquad
R_3 \approx 26{,}665721576411\,V.
$$

O resíduo $\|A\,R-P\|_2$ é menor que $10^{-12}$.


In [ ]:
A = np.array([[17.0, -2.0, -3.0],
              [-5.0, 21.0, -2.0],
              [-5.0, -5.0, 22.0]], dtype=float)
b = np.array([500.0, 200.0, 300.0], dtype=float)

R = seidel(A, b, k=10000, TOL=1e-12)
print('R1 =', R[0])
print('R2 =', R[1])
print('R3 =', R[2])
print('Resíduo:', np.linalg.norm(A @ R - b))


## Atividade 3
Determine o valor da corrente no resistor **R3**, sabendo que $V_1 = V_2 = 100\,V$ e $R_1 = R_2 = R_3 = 10\,\Omega$.

### Circuito equivalente

            R2           R1
     ┌─────/\/\/\───┬──/\/\/\───┐
     │             │            │
     │             │            │
    V2(+)        /\/\/\      V1(+)
     │            R3            │
     │             │            │
     └─────────────┴────────────┘

$$
\begin{bmatrix}
20 & 10 \\
10 & 20
\end{bmatrix}
\begin{Bmatrix}
I_1 \\
I_2
\end{Bmatrix}
=
\begin{Bmatrix}
100 \\
100
\end{Bmatrix}
$$

**Sugestão:** usar o método de Gauss-Seidel.

As correntes de malha são orientadas de modo que, no ramo compartilhado por $R_3$, elas têm o mesmo sentido. Assim, $I_{R3}=I_1+I_2$. A solução do sistema é $I_1=I_2\approx3{,}333333\,A$, portanto:

$$
I_{R3}\approx6{,}666667\,A.
$$


In [ ]:
A = np.array([[20.0, 10.0],
              [10.0, 20.0]], dtype=float)
V = np.array([100.0, 100.0], dtype=float)

I = seidel(A, V, k=10000, TOL=1e-12)
IR3 = I[0] + I[1]
print('I1 =', I[0])
print('I2 =', I[1])
print('Corrente no resistor R3 =', IR3)


## Atividade 4

Num sistema de energia solar, um vetor de equilíbrio energético $E$, na placa absorvente e na placa de vidro, segue o seguinte sistema de equações não lineares nas temperaturas absolutas (em kelvin, $K$):

$$
(T_1^4 + 0,06823T_1) - (T_2^4 + 0,05848T_2) = E_1
$$

$$
(T_1^4 + 0,05848T_1) - (2T_2^4 + 0,11696T_2) = E_2
$$

Calcule o valor aproximado das temperaturas de equilíbrio, sabendo que os valores de equilíbrio energético foram medidos experimentalmente em:  

$E_1 = 0,01753$ e $E_2 = 0,00254$.

**Sugestão:** usar o método de Newton-Raphson.

Aplicando Newton-Raphson a partir de $(T_1^{(0)},T_2^{(0)})=(0{,}3,0{,}2)$, obtêm-se:

$$
T_1\approx0{,}305430071690,\qquad T_2\approx0{,}185261222582.
$$

O resíduo $\|F(T_1,T_2)\|_2$ é menor que $10^{-12}$.


In [ ]:
E1, E2 = 0.01753, 0.00254

def F(x):
    T1, T2 = x
    return np.array([
        (T1**4 + 0.06823 * T1) - (T2**4 + 0.05848 * T2) - E1,
        (T1**4 + 0.05848 * T1) - (2 * T2**4 + 0.11696 * T2) - E2,
    ])

def jacobiana(x):
    T1, T2 = x
    return np.array([
        [4 * T1**3 + 0.06823, -4 * T2**3 - 0.05848],
        [4 * T1**3 + 0.05848, -8 * T2**3 - 0.11696],
    ])

x = np.array([0.3, 0.2], dtype=float)
tol = 1e-12
max_iter = 100

for iteracao in range(1, max_iter + 1):
    passo = np.linalg.solve(jacobiana(x), -F(x))
    x += passo
    if np.linalg.norm(passo, ord=np.inf) <= tol and np.linalg.norm(F(x), ord=np.inf) <= tol:
        break
else:
    raise RuntimeError('Newton-Raphson não convergiu dentro do limite de iterações.')

print(f'T1 = {x[0]:.12f} K')
print(f'T2 = {x[1]:.12f} K')
print(f'Resíduo: {np.linalg.norm(F(x)):.3e}')
print(f'Iterações: {iteracao}')


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 8"
git push origin semana8
```
